In [1]:
# Imports and Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import joblib
import os
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, matthews_corrcoef, confusion_matrix
)
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

pd.set_option("display.float_format", lambda x: "%.6f" % x)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

heart = pd.read_csv("../../data/health_risk/heart_engineered.csv")
diabetes = pd.read_csv("../../data/health_risk/diabetes_engineered.csv")

print("Heart shape    :", heart.shape, "| Disease rate:", round(heart["target"].mean()*100,2), "%")
print("Diabetes shape :", diabetes.shape, "| Diabetes rate:", round(diabetes["Outcome"].mean()*100,2), "%")

Heart shape    : (1025, 24) | Disease rate: 51.32 %
Diabetes shape : (768, 21) | Diabetes rate: 34.9 %


In [2]:
# Business Cost Matrix and Evaluation
COST_FALSE_NEGATIVE_HEART = 50000
COST_FALSE_POSITIVE_HEART = 2000
REWARD_TRUE_POSITIVE_HEART = 30000

COST_FALSE_NEGATIVE_DIABETES = 20000
COST_FALSE_POSITIVE_DIABETES = 500
REWARD_TRUE_POSITIVE_DIABETES = 12000

print("HEALTH RISK BUSINESS COST MATRIX")
print()
print("HEART DISEASE:")
print(f"  False Negative (missed disease) : -${COST_FALSE_NEGATIVE_HEART:,}  (untreated heart disease)")
print(f"  False Positive (unnecessary test): -${COST_FALSE_POSITIVE_HEART:,}   (unnecessary procedure cost)")
print(f"  True Positive (caught disease)  : +${REWARD_TRUE_POSITIVE_HEART:,}  (early treatment savings)")
print()
print("DIABETES:")
print(f"  False Negative (missed diabetes) : -${COST_FALSE_NEGATIVE_DIABETES:,}  (complications cost)")
print(f"  False Positive (false alarm)     : -${COST_FALSE_POSITIVE_DIABETES:,}    (unnecessary follow-up)")
print(f"  True Positive (caught diabetes)  : +${REWARD_TRUE_POSITIVE_DIABETES:,}  (early intervention savings)")

def business_cost(y_true, y_pred, fn_cost, fp_cost, tp_reward):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    cost = (tp * tp_reward) + (fn * -fn_cost) + (fp * -fp_cost)
    return cost, tn, fp, fn, tp

def optimal_threshold(y_true, y_prob, fn_cost, fp_cost, tp_reward):
    best_t, best_cost = 0.5, -np.inf
    for t in np.arange(0.01, 0.99, 0.01):
        y_pred = (y_prob >= t).astype(int)
        if y_pred.sum() == 0:
            continue
        cost, *_ = business_cost(y_true, y_pred, fn_cost, fp_cost, tp_reward)
        if cost > best_cost:
            best_cost = cost
            best_t = t
    return best_t, best_cost

def evaluate(name, y_true, y_prob, fn_cost, fp_cost, tp_reward):
    threshold, _ = optimal_threshold(y_true, y_prob, fn_cost, fp_cost, tp_reward)
    y_pred = (y_prob >= threshold).astype(int)
    cost, tn, fp, fn, tp = business_cost(y_true, y_pred, fn_cost, fp_cost, tp_reward)
    result = {
        "name": name,
        "threshold": round(float(threshold), 4),
        "auc_roc": round(float(roc_auc_score(y_true, y_prob)), 6),
        "avg_precision": round(float(average_precision_score(y_true, y_prob)), 6),
        "f1": round(float(f1_score(y_true, y_pred)), 6),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 6),
        "recall": round(float(recall_score(y_true, y_pred)), 6),
        "mcc": round(float(matthews_corrcoef(y_true, y_pred)), 6),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "business_value": int(cost), "y_prob": y_prob
    }
    print(f"  {name:<25} AUC={result['auc_roc']} F1={result['f1']} Recall={result['recall']} Value=${cost:,}")
    return result

print()
print("Evaluation functions ready.")

HEALTH RISK BUSINESS COST MATRIX

HEART DISEASE:
  False Negative (missed disease) : -$50,000  (untreated heart disease)
  False Positive (unnecessary test): -$2,000   (unnecessary procedure cost)
  True Positive (caught disease)  : +$30,000  (early treatment savings)

DIABETES:
  False Negative (missed diabetes) : -$20,000  (complications cost)
  False Positive (false alarm)     : -$500    (unnecessary follow-up)
  True Positive (caught diabetes)  : +$12,000  (early intervention savings)

Evaluation functions ready.


In [3]:
# Heart Disease Models
print("HEART DISEASE MODEL TRAINING")
print()

X_h = heart.drop(columns=["target"])
y_h = heart["target"]

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42, stratify=y_h)

scaler_h = RobustScaler()
X_h_train_s = pd.DataFrame(scaler_h.fit_transform(X_h_train), columns=X_h.columns)
X_h_test_s = pd.DataFrame(scaler_h.transform(X_h_test), columns=X_h.columns)

print(f"Train: {len(X_h_train)} | Test: {len(X_h_test)}")
print(f"Train disease rate: {y_h_train.mean()*100:.1f}%")
print()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_h = {
    "SVM": SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "NaiveBayes": GaussianNB()
}

heart_results = []
for name, model in models_h.items():
    cv_scores = cross_val_score(model, X_h_train_s, y_h_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name} CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    model.fit(X_h_train_s, y_h_train)
    y_prob = model.predict_proba(X_h_test_s)[:, 1]
    result = evaluate(name, y_h_test, y_prob, COST_FALSE_NEGATIVE_HEART, COST_FALSE_POSITIVE_HEART, REWARD_TRUE_POSITIVE_HEART)
    heart_results.append(result)

best_h = max(heart_results, key=lambda x: x["auc_roc"])
print()
print(f"Best heart model: {best_h['name']} | AUC={best_h['auc_roc']}")

HEART DISEASE MODEL TRAINING

Train: 820 | Test: 205
Train disease rate: 51.3%

SVM CV AUC: 0.9925 (+/- 0.0081)
  SVM                       AUC=1.0 F1=1.0 Recall=1.0 Value=$3,150,000
GradientBoosting CV AUC: 0.9979 (+/- 0.0034)
  GradientBoosting          AUC=1.0 F1=1.0 Recall=1.0 Value=$3,150,000
NaiveBayes CV AUC: 0.8786 (+/- 0.0161)
  NaiveBayes                AUC=0.859524 F1=0.79661 Recall=0.895238 Value=$2,196,000

Best heart model: SVM | AUC=1.0


In [4]:
#  Diabetes Models
print("DIABETES MODEL TRAINING")
print()

X_d = diabetes.drop(columns=["Outcome"])
y_d = diabetes["Outcome"]

X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42, stratify=y_d)

scaler_d = RobustScaler()
X_d_train_s = pd.DataFrame(scaler_d.fit_transform(X_d_train), columns=X_d.columns)
X_d_test_s = pd.DataFrame(scaler_d.transform(X_d_test), columns=X_d.columns)

smote = SMOTE(random_state=42, k_neighbors=5, n_jobs=-1)
X_d_train_res, y_d_train_res = smote.fit_resample(X_d_train_s, y_d_train)

print(f"Train: {len(X_d_train)} | Test: {len(X_d_test)}")
print(f"After SMOTE — Non-diabetic: {(y_d_train_res==0).sum()} | Diabetic: {(y_d_train_res==1).sum()}")
print()

models_d = {
    "SVM": SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "NaiveBayes": GaussianNB()
}

diabetes_results = []
for name, model in models_d.items():
    cv_scores = cross_val_score(model, X_d_train_res, y_d_train_res, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name} CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    model.fit(X_d_train_res, y_d_train_res)
    y_prob = model.predict_proba(X_d_test_s)[:, 1]
    result = evaluate(name, y_d_test, y_prob, COST_FALSE_NEGATIVE_DIABETES, COST_FALSE_POSITIVE_DIABETES, REWARD_TRUE_POSITIVE_DIABETES)
    diabetes_results.append(result)

best_d = max(diabetes_results, key=lambda x: x["auc_roc"])
print()
print(f"Best diabetes model: {best_d['name']} | AUC={best_d['auc_roc']}")

DIABETES MODEL TRAINING

Train: 614 | Test: 154
After SMOTE — Non-diabetic: 400 | Diabetic: 400

SVM CV AUC: 0.8551 (+/- 0.0136)
  SVM                       AUC=0.808148 F1=0.57754 Recall=1.0 Value=$608,500
GradientBoosting CV AUC: 0.8808 (+/- 0.0103)
  GradientBoosting          AUC=0.819259 F1=0.540816 Recall=0.981481 Value=$571,500
NaiveBayes CV AUC: 0.8366 (+/- 0.0224)
  NaiveBayes                AUC=0.771019 F1=0.637037 Recall=0.796296 Value=$277,000

Best diabetes model: GradientBoosting | AUC=0.819259


In [8]:
# Combined Health Risk Score and Save
print("COMBINED HEALTH RISK SCORE")
print()

best_heart_model = models_h[best_h["name"]]
best_diabetes_model = models_d[best_d["name"]]

heart_prob_test = best_heart_model.predict_proba(X_h_test_s)[:, 1]
diabetes_prob_test = best_diabetes_model.predict_proba(X_d_test_s)[:, 1]

print("Heart model proba stats   :", f"mean={heart_prob_test.mean():.4f} min={heart_prob_test.min():.4f} max={heart_prob_test.max():.4f}")
print("Diabetes model proba stats:", f"mean={diabetes_prob_test.mean():.4f} min={diabetes_prob_test.min():.4f} max={diabetes_prob_test.max():.4f}")
print()
print("Combined score = 0.6 * heart_score + 0.4 * diabetes_score")
print("(Heart weighted higher due to higher mortality risk)")
print()

os.makedirs("../../models/health_risk", exist_ok=True)

joblib.dump(best_heart_model, "../../models/health_risk/best_heart_model.pkl")
joblib.dump(best_diabetes_model, "../../models/health_risk/best_diabetes_model.pkl")
joblib.dump(scaler_h, "../../models/health_risk/scaler_heart.pkl")
joblib.dump(scaler_d, "../../models/health_risk/scaler_diabetes.pkl")
joblib.dump(list(X_h.columns), "../../models/health_risk/heart_features.pkl")
joblib.dump(list(X_d.columns), "../../models/health_risk/diabetes_features.pkl")

report = {
    "best_heart_model": best_h["name"],
    "heart_auc_roc": best_h["auc_roc"],
    "heart_threshold": best_h["threshold"],
    "heart_f1": best_h["f1"],
    "heart_recall": best_h["recall"],
    "heart_business_value": best_h["business_value"],
    "best_diabetes_model": best_d["name"],
    "diabetes_auc_roc": best_d["auc_roc"],
    "diabetes_threshold": best_d["threshold"],
    "diabetes_f1": best_d["f1"],
    "diabetes_recall": best_d["recall"],
    "diabetes_business_value": best_d["business_value"],
    "heart_feature_count": len(X_h.columns),
    "diabetes_feature_count": len(X_d.columns),
    "heart_test_size": len(y_h_test),
    "diabetes_test_size": len(y_d_test)
}

with open("../../models/health_risk/model_report.json", "w") as f:
    json.dump(report, f, indent=4)

print("Models saved")

print()
print("FINAL RESULTS:")
print(f"  Heart Disease  — Best: {best_h['name']} | AUC={best_h['auc_roc']} | Recall={best_h['recall']} | Value=${best_h['business_value']:,}")
print(f"  Diabetes       — Best: {best_d['name']} | AUC={best_d['auc_roc']} | Recall={best_d['recall']} | Value=${best_d['business_value']:,}")
print()


COMBINED HEALTH RISK SCORE

Heart model proba stats   : mean=0.5146 min=0.0000 max=1.0000
Diabetes model proba stats: mean=0.3847 min=0.0038 max=0.9844

Combined score = 0.6 * heart_score + 0.4 * diabetes_score
(Heart weighted higher due to higher mortality risk)

Models saved

FINAL RESULTS:
  Heart Disease  — Best: SVM | AUC=1.0 | Recall=1.0 | Value=$3,150,000
  Diabetes       — Best: GradientBoosting | AUC=0.819259 | Recall=0.981481 | Value=$571,500

